## **1. Kiểm tra file jsonl**

In [4]:
import json
from collections import Counter
from urllib.parse import urlparse

FILE = "../../data/01_discovered/2026-03.jsonl"

urls = []
queries = []
domains = []

with open(FILE, "r", encoding="utf-8") as f:
    for line in f:

        record = json.loads(line)

        query_id = record.get("query_id")

        for r in record["results"]:
            url = r["url"]

            urls.append(url)
            queries.append(query_id)

            domain = urlparse(url).netloc
            domains.append(domain)

# =========================
# BASIC STATS
# =========================

total_urls = len(urls)
unique_urls = len(set(urls))

print("Total URLs:", total_urls)
print("Unique URLs:", unique_urls)
print("Duplicate URLs:", total_urls - unique_urls)

# =========================
# DUPLICATE LINKS
# =========================

url_counter = Counter(urls)

duplicates = {u:c for u,c in url_counter.items() if c > 1}

print("\nNumber of duplicated URLs:", len(duplicates))

print("\nTop duplicated URLs:")
for u,c in sorted(duplicates.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(c, u)

# =========================
# DOMAIN DISTRIBUTION
# =========================

domain_counter = Counter(domains)

print("\nTop domains:")
for d,c in domain_counter.most_common(10):
    print(c, d)

# =========================
# QUERY DISTRIBUTION
# =========================

query_counter = Counter(queries)

print("\nQuery counts:")
for q,c in query_counter.most_common():
    print(c, q)

Total URLs: 430
Unique URLs: 148
Duplicate URLs: 282

Number of duplicated URLs: 97

Top duplicated URLs:
8 https://laodong.vn/thi-truong/gia-ca-phe-hom-nay-53-bat-tang-manh-me-1664241.ldo
8 https://vov.vn/thi-truong/gia-ca-phe-hom-nay-73-gia-ca-phe-trong-nuoc-cao-nhat-96000-dongkg-post1273567.vov
8 https://trangtraiviet.danviet.vn/gia-ca-phe-tang-xung-dot-o-trung-dong-lam-gian-doan-van-chuyen-ca-phe-tu-viet-nam-d1408010.html
8 https://vov.vn/thi-truong/gia-ca-phe-hom-nay-103-gia-ca-phe-trong-nuoc-trung-binh-o-muc-96600-dongkg-post1274295.vov
8 https://vietnambiz.vn/gia-ca-phe-hom-nay-113-hai-san-dong-loat-giam-trong-nuoc-ve-duoi-96000-dongkg-202631165850654.htm
8 https://trangtraiviet.danviet.vn/gia-ca-phe-giam-manh-ca-phe-trong-nuoc-trong-nuoc-cung-dieu-chinh-sau-giai-doan-tang-manh-d1409511.html
7 https://vov.vn/thi-truong/gia-ca-phe/gia-ca-phe-hom-nay-43-gia-ca-phe-robusta-giam-gia-ca-phe-arabica-tang-post1272738.vov
7 https://trangtraiviet.danviet.vn/gia-ca-phe-giam-tro-lai-nha-da

## **2. Kiểm tra có crawl đủ URL không**

In [13]:
import json
from pathlib import Path

DISCOVERY = Path("../../data/01_discovered/2026-03.jsonl")
RAW = Path("../../data/02_rawhtml")

urls = set()

with open(DISCOVERY, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        for r in record["results"]:
            urls.add(r["url"])

html_files = list(RAW.glob("**/*.html"))

print("Unique URLs:", len(urls))
print("HTML files:", len(html_files))
print("Missing:", len(urls) - len(html_files))

Unique URLs: 148
HTML files: 129
Missing: 19


## **3. Kiểm tra các HTML có rỗng không**

In [14]:
bad = []

for f in html_files:
    if f.stat().st_size < 2000:
        bad.append(f)

print("Bad HTML:", len(bad))
bad[:10]

Bad HTML: 0


[]

## **4. Kiểm tra metadata có đầy đủ không**

In [15]:
meta_files = list(RAW.glob("**/*.json"))

print("HTML:", len(html_files))
print("META:", len(meta_files))

HTML: 129
META: 129


## **5. Kiểm tra domain distribution** 

In [16]:
from urllib.parse import urlparse
from collections import Counter

domains = []

with open(DISCOVERY, "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        for r in record["results"]:
            domains.append(urlparse(r["url"]).netloc)

Counter(domains).most_common(10)

[('baomoi.com', 96),
 ('trangtraiviet.danviet.vn', 55),
 ('vov.vn', 50),
 ('laodong.vn', 45),
 ('nld.com.vn', 28),
 ('vietnambiz.vn', 28),
 ('thoibaotaichinhvietnam.vn', 25),
 ('www.instagram.com', 15),
 ('giacaphe.com', 10),
 ('cafef.vn', 7)]